# 03 — Structural Model: Direct Predictors of Critical Thinking Engagement

**Input:** `items_final_measurement_model.csv` and `final_items.json` from notebook 2.

This is the central test of the study: does AI literacy and metacognitive
awareness predict critical thinking engagement independently of AI usage and
cognitive offloading? One-step latent SEM is used (rather than two-step,
composite-score regression) so that measurement error is properly accounted
for in the structural estimates.

In [ ]:
!pip install semopy

In [ ]:
import pandas as pd
import numpy as np
import json
from semopy import Model

df = pd.read_csv("items_final_measurement_model.csv", index_col=0)
final_items = json.load(open("final_items.json"))
print("N =", len(df))

desc = "".join(f"{c} =~ " + " + ".join(v) + "\n" for c, v in final_items.items())
struct = "CTE ~ AIU + CO + AIL + MMR\n"
m = Model(desc + struct)
m.fit(df)

ins = m.inspect(std_est=True)
for col in ["Estimate", "Est. Std", "Std. Err", "z-value", "p-value"]:
    ins[col] = pd.to_numeric(ins[col], errors="coerce")
paths = ins[(ins["op"] == "~") & (ins["lval"] == "CTE")]

print("Structural paths predicting CTE:\n")
for _, r in paths.iterrows():
    print(f"  {r['rval']:<5} beta={r['Est. Std']:>7.3f}  SE={r['Std. Err']:>7.3f}  "
          f"z={r['z-value']:>7.2f}  p={r['p-value']:>8.4f}")
resid = ins[(ins["op"] == "~~") & (ins["lval"] == "CTE") & (ins["rval"] == "CTE")]["Est. Std"]
r2 = 1 - float(resid.iloc[0])
print(f"\n  R-squared = {r2:.4f}")

## Multicollinearity check (VIF)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

comp = pd.DataFrame({c: df[items].mean(axis=1) for c, items in final_items.items()})
X = comp[["AIU", "CO", "AIL", "MMR"]].values
print("Variance inflation factors (should all be well below 5):")
for i, name in enumerate(["AIU", "CO", "AIL", "MMR"]):
    print(f"  {name}: {variance_inflation_factor(X, i):.2f}")

comp.to_csv("composite_scores.csv")

## Bootstrap confidence intervals

A brief robustness check on the two significant paths, since Wald-type standard
errors rely on large-sample theory. This resamples respondents with replacement
and refits the full model — expect this cell to take a minute or two.

In [ ]:
np.random.seed(42)
N = len(df)
n_boot = 1000
boot_betas = {p: [] for p in ["AIU", "CO", "AIL", "MMR"]}
failed = 0
for b in range(n_boot):
    idx = np.random.choice(N, N, replace=True)
    dfb = df.iloc[idx].reset_index(drop=True)
    try:
        mb = Model(desc + struct)
        mb.fit(dfb)
        insb = mb.inspect(std_est=True)
        insb["Est. Std"] = pd.to_numeric(insb["Est. Std"], errors="coerce")
        pb = insb[(insb["op"] == "~") & (insb["lval"] == "CTE")]
        row = dict(zip(pb["rval"], pb["Est. Std"]))
        if all(p in row and pd.notna(row[p]) for p in boot_betas):
            for p in boot_betas:
                boot_betas[p].append(row[p])
        else:
            failed += 1
    except Exception:
        failed += 1

print(f"{n_boot} resamples, {failed} failed to converge\n")
for p, vals in boot_betas.items():
    arr = np.array(vals)
    lo, hi = np.percentile(arr, [2.5, 97.5])
    print(f"  {p}: 95% CI = [{lo:+.3f}, {hi:+.3f}]")

**Interpretation:** if AIL and MMR's intervals exclude zero and AIU/CO's include
zero, the bootstrap corroborates the Wald-based result above.

## Supplementary check: AIL × Metacognition interaction

A secondary, exploratory question: does literacy's association with critical
thinking depend on metacognitive awareness? Tested on composite scores; treat
this as exploratory regardless of outcome, since it was not the primary
confirmatory test.

In [ ]:
import statsmodels.formula.api as smf

comp_c = comp.copy()
for c in ["AIU", "CO", "AIL", "MMR"]:
    comp_c[c + "_c"] = comp_c[c] - comp_c[c].mean()

m_base = smf.ols("CTE ~ AIU_c + CO_c + AIL_c + MMR_c", comp_c).fit()
m_int = smf.ols("CTE ~ AIU_c + CO_c + AIL_c + MMR_c + AIL_c:MMR_c", comp_c).fit()
b = m_int.params["AIL_c:MMR_c"]; p = m_int.pvalues["AIL_c:MMR_c"]
print(f"AIL x MMR interaction: b={b:.4f}  p={p:.4f}  "
      f"(delta R2 = {m_int.rsquared - m_base.rsquared:.4f})")
print("\nReport this result as-is, whether significant or not -- it was pre-specified")
print("as exploratory and its outcome does not affect the main structural finding above.")

**Next notebook:** `04_invariance.ipynb`